# Why does parity fail at frame 7 on a T4?

`gpu_parity.py` reports `range_image` differs at frame 7 on a Tesla T4, while
CPU and CUDA agree bit for bit on the laptop RTX 5050.

The key is `(pix << 49) | (range_bits << 18) | point_index`. The index is unique,
so every key is unique, the sort is a total order, and the winner per pixel is
deterministic **by construction** — ties cannot be the cause. The bit budget is
guarded at construction, so it is not an overflow either.

That leaves the float arithmetic that *produces* the key. This asks which:

* **(a) the winner moved** — the same points land in the pixel, but `r` differs by
  an ULP and reorders them; or
* **(b) a point moved pixels** — `az`/`el` differ by an ULP right at a bin edge,
  so `floor()` lands in a different bin.

They have different fixes, so the distinction is the whole point.

In [ ]:
# 1. The machine. This is the whole point of the notebook -- a DIFFERENT card.
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv
import torch
print("torch", torch.__version__, "| cuda", torch.version.cuda)
assert torch.cuda.is_available(), "no GPU -- set Accelerator to GPU in the sidebar"
name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"{name}, {vram:.1f} GB")
assert vram > 12, f"{name} has {vram:.1f} GB -- the 16 GB contention test needs more than the laptop's 8"

In [ ]:
# 2. The repo. Public, so no token.
%cd /kaggle/working
!rm -rf vrgrid-26
!git clone -q https://github.com/Stxtics03/vrgrid-26.git
%cd /kaggle/working/vrgrid-26
!git log --oneline -1
!pip -q install -e . 2>&1 | tail -2
try:
    import cupy; print("cupy", cupy.__version__, "(preinstalled)")
except ImportError:
    !pip -q install cupy-cuda12x

In [ ]:
# 4. Assemble the asset tree vrgrid expects. The mirror supplies sequences/; the
#    bundle supplies poses/ and the checkpoint.
#    loader.py uses the OFFICIAL KITTI GT poses at poses/<seq>.txt, NOT the
#    SemanticKITTI SLAM poses inside sequences/<seq>/poses.txt. Do not substitute
#    one for the other -- they are different quantities and the swap is silent.
import os
from pathlib import Path

# Print what actually mounted before asserting anything. A dataset that is
# attached but empty looks identical to one that is missing, three cells later.
IN = Path("/kaggle/input")
for root, dirs, files in os.walk(IN):
    depth = root.replace(str(IN), "").count(os.sep)
    if depth > 3:
        dirs[:] = []
        continue
    print("  " * depth + os.path.basename(root) + "/")

def find(child, maxdepth=4):
    """The mounted dataset directory containing <child>.

    Kaggle nests these as /kaggle/input/datasets/<owner>/<slug>/, so a
    top-level scan finds only "datasets". Breadth-first with a depth cap --
    never rglob, the mirror alone is 96 GB and 66k files.
    """
    frontier = [IN]
    for _ in range(maxdepth):
        nxt = []
        for d in frontier:
            if (d / child).is_dir():
                return d
            try:
                nxt += [k for k in d.iterdir() if k.is_dir()]
            except (PermissionError, OSError):
                pass
        frontier = nxt
    return None

MIRROR = find("sequences")
BUNDLE = find("poses")
print("\nMIRROR:", MIRROR, "\nBUNDLE:", BUNDLE)
assert MIRROR, "no mounted dataset contains sequences/"
assert BUNDLE, "no mounted dataset contains poses/ -- check the Input panel"

A = Path("/kaggle/working/assets")
(A / "dataset").mkdir(parents=True, exist_ok=True)
for src, dst in [(MIRROR / "sequences", A / "dataset/sequences"),
                 (BUNDLE / "poses",     A / "dataset/poses"),
                 (BUNDLE / "checkpoints", A / "checkpoints")]:
    assert src.exists(), f"missing: {src}"
    if not dst.exists():
        dst.symlink_to(src)

os.environ["VRGRID_ASSETS"] = str(A)
os.environ["VRGRID_DATA_ROOT"] = str(A / "dataset")
os.environ["VRGRID_FRNET_CHECKPOINT"] = str(A / "checkpoints/frnet-semantickitti_seg.pth")
for k in ("VRGRID_ASSETS", "VRGRID_DATA_ROOT", "VRGRID_FRNET_CHECKPOINT"):
    print(f"{k}={os.environ[k]}")


In [ ]:
import pathlib
pathlib.Path('/kaggle/working/vrgrid-26/diag_frame7.py').write_text(r'''
"""Why range_image diverges between CPU and CUDA on a T4, at frame 7.

Run as a SCRIPT, not imported into the notebook kernel: vrgrid is assembled by
setuptools from several source directories (include/vrgrid, src/grid, src/gpu,
...), so it only exists after `pip install -e .`, and a notebook kernel that
started before that install will never see it.
"""
import warnings, numpy as np
from vrgrid.gpu.device import DevicePerception
from vrgrid.perception import ground, loader, semantics
from vrgrid.run.__main__ import perceive

# Frames 0..7 IN ORDER: the ground estimator and Patchwork++ are stateful, so
# frame 7 reached by skipping is not frame 7 reached by replay.
perception = DevicePerception()
ground.reset_estimator()
bad = None
for i, (points, labels, pose) in enumerate(loader.scans("08", max_frames=12, start_frame=0)):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        gres = ground.segment_ground_or_fallback(
            points, semantics.semantic_labels(labels), use_patchworkpp=True)
        host = perceive(points, labels, pose, "08", i, ground_result=gres)
        dev  = perceive(points, labels, pose, "08", i,
                        perception=perception, ground_result=gres)
    hr, dr = np.asarray(host.range_image), np.asarray(dev.range_image)
    ok = np.array_equal(hr, dr, equal_nan=True)
    print(f"frame {i}: range_image identical = {ok}", flush=True)
    if not ok:
        bad = (i, points, host, dev, hr, dr); break

if bad is None:
    print("\nNo divergence in 12 frames -- not reproducible this run."); raise SystemExit(0)

i, points, host, dev, hr, dr = bad
hi, di = np.asarray(host.inverse_index), np.asarray(dev.inverse_index)
H, W = hr.shape[0], hr.shape[1]
same = (hr == dr) | (np.isnan(hr) & np.isnan(dr))
pixdiff = ~same.all(axis=2) if hr.ndim == 3 else ~same
vs, us = np.nonzero(pixdiff)
print(f"\n=== frame {i}: {len(vs)} differing pixels of {H*W} ({100*len(vs)/(H*W):.4f}%) ===\n")

d_theta, d_phi, phi_max = perception.d_theta, perception.d_phi, perception.phi_max

def proj(p, dt):
    x, y, z = dt(p[0]), dt(p[1]), dt(p[2])
    r  = np.sqrt(x*x + y*y + z*z)
    az = np.arctan2(np.float64(y), np.float64(x))
    el = np.arcsin(np.clip(np.float64(z)/np.float64(r), -1.0, 1.0))
    u  = int(np.floor((np.float32(az) + np.float32(np.pi)) / np.float32(d_theta))) % W
    v  = int(np.floor((phi_max - el) / d_phi))
    return r, u, v

moved_pixel = winner_changed = 0
for v, u in list(zip(vs, us)):
    a, b = int(hi[v, u]), int(di[v, u])
    if (a < 0) != (b < 0): moved_pixel += 1
    elif a != b:           winner_changed += 1

print(f"VERDICT COUNTS")
print(f"  (a) same pixel, winner changed : {winner_changed}")
print(f"  (b) a point moved pixels       : {moved_pixel}")
print(f"  (c) same winner, value differs : {len(vs) - moved_pixel - winner_changed}\n")

for v, u in list(zip(vs, us))[:8]:
    a, b = int(hi[v, u]), int(di[v, u])
    tag = ("(b) POINT MOVED PIXELS" if (a < 0) != (b < 0)
           else "(a) WINNER CHANGED" if a != b else "(c) SAME WINNER, VALUE DIFFERS")
    hv = hr[v, u, 0] if hr.ndim == 3 else hr[v, u]
    dv = dr[v, u, 0] if dr.ndim == 3 else dr[v, u]
    print(f"pixel (v={v}, u={u})  cpu_idx={a} gpu_idx={b}  -> {tag}")
    print(f"    cpu range {hv!r}   gpu range {dv!r}")
    for nm, idx in (("cpu", a), ("gpu", b)):
        if idx < 0: continue
        p = points[idx]
        r32, u32, v32 = proj(p, np.float32)
        r64, u64, v64 = proj(p, np.float64)
        print(f"    {nm} pt {idx}: xyz=({p[0]:.6f},{p[1]:.6f},{p[2]:.6f})")
        print(f"        r f32={r32!r} f64={r64!r} delta={float(r64)-float(r32):.3e}")
        print(f"        bin f32=({v32},{u32}) f64=({v64},{u64})"
              + ("   <-- BIN DEPENDS ON PRECISION" if (u32, v32) != (u64, v64) else ""))
    print()

# How near a bin edge are the culprits? If they sit within an ULP of an integer
# boundary then the binning itself is the problem and tie-breaking cannot fix it.
edge = []
for v, u in zip(vs, us):
    for idx in (int(hi[v, u]), int(di[v, u])):
        if idx < 0: continue
        x, y, z = map(np.float64, points[idx][:3])
        r  = np.sqrt(x*x + y*y + z*z)
        az = np.arctan2(y, x); el = np.arcsin(np.clip(z/r, -1, 1))
        tu = (az + np.pi)/d_theta; tv = (phi_max - el)/d_phi
        edge.append((abs(tu - round(tu)), abs(tv - round(tv))))
if edge:
    e = np.array(edge)
    print(f"=== {len(e)} offending points, distance to nearest bin edge ===")
    print(f"  azimuth   min {e[:,0].min():.3e}  median {np.median(e[:,0]):.3e}  "
          f"within 1e-6: {(e[:,0] < 1e-6).sum()}")
    print(f"  elevation min {e[:,1].min():.3e}  median {np.median(e[:,1]):.3e}  "
          f"within 1e-6: {(e[:,1] < 1e-6).sum()}")
''')
print('wrote diag_frame7.py')

In [ ]:
# Run as a subprocess from inside the repo -- the only context where the
# editable install of vrgrid is importable.
!cd /kaggle/working/vrgrid-26 && python diag_frame7.py 2>&1 | tee /kaggle/working/frame7.log